<a href="https://colab.research.google.com/github/gaelosvaldor-star/Simulaci-n-2-/blob/main/Gael_Osvaldo_Rosales_SIR_PyMC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo SIR: estimación bayesiana con PyMC
### Simulación II · ESFM · 2027-1
**Alumno:** Gael Osvaldo Rosales de la Cruz  
**Fecha:** 17 de octubre de 2026  




## 1. Planteamiento y ecuaciones

Dividimos una población cerrada de tamaño $N$ en tres grupos: susceptibles $S(t)$, infectados $I(t)$ y recuperados $R(t)$. El modelo determinista es

$$\frac{dS}{dt}=-\beta\frac{SI}{N},\qquad
\frac{dI}{dt}=\beta\frac{SI}{N}-\gamma I,\qquad
\frac{dR}{dt}=\gamma I.$$

Aquí $\beta>0$ es la tasa de transmisión y $\gamma>0$ la tasa de recuperación, ambas expresadas por día. Se cumple $S+I+R=N$. El número reproductivo básico es $R_0=\beta/\gamma$ (no confundir con el compartimento $R(t)$). Si $R_0>1$, al comienzo puede aumentar el número de infectados cuando casi todos son susceptibles.

**Objetivo:** inferir $\beta$ y $\gamma$ a partir de mediciones ruidosas del número de infectados activos y describir su incertidumbre.

## 2. Bibliotecas y reproducibilidad

Los datos se generan con una semilla fija. PyMC construye el modelo probabilístico y ArviZ permite diagnosticar las cadenas de Markov.

In [ ]:
# Si trabajas en Google Colab y falta alguna biblioteca, descomenta:
# %pip -q install "pymc>=5" arviz scipy matplotlib pandas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt
import arviz as az
from scipy.integrate import solve_ivp

SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)
print("PyMC:", pm.__version__, "| ArviZ:", az.__version__)

## 3. Datos sintéticos

En ausencia de una serie epidemiológica suministrada por la tarea, se simula una epidemia de referencia con $N=10\,000$, $I(0)=15$, $R(0)=0$, $\beta=0.30$ y $\gamma=0.11$. El algoritmo de observación agrega ruido multiplicativo lognormal para representar mediciones imperfectas de infectados activos. **Estas tasas conocidas se usan exclusivamente para crear datos de prueba; en el ajuste se consideran desconocidas.**

In [ ]:
N = 10_000
I0, R_inicial = 15.0, 0.0
S0 = N - I0 - R_inicial
beta_real, gamma_real = 0.30, 0.11
dias = np.arange(0, 51, dtype=float)
sigma_obs = 0.12  # desviación estándar del error en escala logarítmica

def sir_deriv(t, y, beta, gamma):
    S, I, R = y
    contagios = beta * S * I / N
    recuperaciones = gamma * I
    return [-contagios, contagios - recuperaciones, recuperaciones]

solucion = solve_ivp(sir_deriv, (dias[0], dias[-1]),
                     [S0, I0, R_inicial], t_eval=dias,
                     args=(beta_real, gamma_real), rtol=1e-9, atol=1e-9)
assert solucion.success
S_real, I_real, R_real = solucion.y
infectados_obs = np.exp(np.log(I_real + 1.0) + rng.normal(0, sigma_obs, len(dias))) - 1
infectados_obs = np.maximum(infectados_obs, 0)

datos = pd.DataFrame({"dia": dias.astype(int), "infectados_observados": infectados_obs})
display(datos.head())
plt.figure(figsize=(10, 4))
plt.plot(dias, I_real, label="Infectados sin ruido", linewidth=2)
plt.scatter(dias, infectados_obs, s=18, alpha=.7, label="Observaciones sintéticas")
plt.xlabel("Día"); plt.ylabel("Infectados activos")
plt.title("Serie sintética para el ajuste"); plt.legend(); plt.grid(alpha=.25); plt.show()

## 4. Modelo estadístico bayesiano

Usamos la factorización de Bayes:

$$p(\beta,\gamma\mid y)\ \propto\ p(y\mid\beta,\gamma)\,p(\beta)\,p(\gamma).$$

- **Previas:** $\beta\sim\operatorname{LogNormal}(\log(0.30),0.55)$ y $\gamma\sim\operatorname{LogNormal}(\log(0.11),0.55)$; garantizan tasas positivas sin fijarlas.
- **Verosimilitud:** $\log(y_t+1)\sim\mathcal N(\log(I_t(\beta,\gamma)+1),\sigma_{obs}^2)$, con $\sigma_{obs}=0.12$ conocida en esta simulación.
- **Integración del SIR:** implementamos Runge–Kutta de orden 4 (RK4) con paso de un día usando operaciones diferenciables de PyTensor. Así, NUTS puede explorar la distribución posterior sin una función externa opaca.

El paso numérico aproxima la solución continua. La discretización se revisa contra `solve_ivp` antes del ajuste.

In [ ]:
# RK4 puro con NumPy, para evaluar el error numérico.
def sir_rk4_np(beta, gamma, pasos=50):
    estado = np.array([S0, I0, R_inicial], dtype=float)
    historia = [estado.copy()]
    for _ in range(pasos):
        def f(v):
            S, I, R = v
            q = beta * S * I / N
            g = gamma * I
            return np.array([-q, q-g, g])
        k1=f(estado); k2=f(estado+k1/2); k3=f(estado+k2/2); k4=f(estado+k3)
        estado = estado+(k1+2*k2+2*k3+k4)/6
        historia.append(estado.copy())
    return np.stack(historia)

error = np.max(np.abs(sir_rk4_np(beta_real,gamma_real) - solucion.y.T))
print(f"Error máximo frente a solve_ivp: {error:.5f} personas")
assert error < 1.0, "Revisar tamaño de paso o parámetros."

In [ ]:
# Integrador RK4 simbólico: PyTensor conserva derivadas respecto a beta y gamma.
def sir_rk4_pt(beta, gamma, pasos=50):
    estado = pt.stack([pt.as_tensor_variable(S0),
                       pt.as_tensor_variable(I0),
                       pt.as_tensor_variable(R_inicial)])
    estados = [estado]
    def f(v):
        S, I, R = v[0], v[1], v[2]
        q = beta * S * I / N
        g = gamma * I
        return pt.stack([-q, q-g, g])
    for _ in range(pasos):
        k1 = f(estado)
        k2 = f(estado + k1 / 2)
        k3 = f(estado + k2 / 2)
        k4 = f(estado + k3)
        estado = estado + (k1 + 2*k2 + 2*k3 + k4) / 6
        estados.append(estado)
    return pt.stack(estados)

with pm.Model() as modelo_sir:
    beta = pm.LogNormal("beta", mu=np.log(0.30), sigma=0.55)
    gamma = pm.LogNormal("gamma", mu=np.log(0.11), sigma=0.55)
    trayectoria = sir_rk4_pt(beta, gamma, pasos=len(dias)-1)
    I_modelo = trayectoria[:, 1]
    R0_basico = pm.Deterministic("R0_basico", beta / gamma)
    pm.Normal("log_infectados", mu=pt.log(I_modelo + 1), sigma=sigma_obs,
              observed=np.log(infectados_obs + 1))

# Muestreo: cada cadena requiere calentamiento y muestras posteriores.
# Puede tardar algunos minutos en la primera ejecución debido a la compilación.
with modelo_sir:
    idata = pm.sample(draws=800, tune=800, chains=2, cores=1,
                      random_seed=SEMILLA, target_accept=0.92,
                      return_inferencedata=True)
print("Muestreo completado.")

## 5. Resumen y diagnóstico de convergencia

La media y los intervalos HDI del 94% resumen la distribución posterior. Para verificar la estabilidad, se inspecciona `r_hat` (idealmente cercano a 1), el tamaño efectivo de muestra (`ess_bulk`) y los trazos de las cadenas; valores inadecuados exigirían un muestreo más largo o reformular el modelo.

In [ ]:
resumen = az.summary(idata, var_names=["beta", "gamma", "R0_basico"], hdi_prob=0.94)
display(resumen)
az.plot_trace(idata, var_names=["beta", "gamma", "R0_basico"])
plt.tight_layout(); plt.show()
az.plot_posterior(idata, var_names=["beta", "gamma", "R0_basico"],
                  hdi_prob=0.94, ref_val=[beta_real, gamma_real, beta_real/gamma_real])
plt.tight_layout(); plt.show()
print("R-hat máximo:", resumen['r_hat'].max())
print("ESS bulk mínimo:", resumen['ess_bulk'].min())

## 6. Ajuste de trayectorias y banda creíble

Calculamos las trayectorias SIR para una submuestra de parámetros de la distribución posterior. La franja entre los percentiles 3 y 97 es una **banda creíble del 94% para la trayectoria latente**. No debe interpretarse como un intervalo de predicción para nuevas observaciones: este último debe incorporar también el error de medición.

In [ ]:
betas = idata.posterior["beta"].values.reshape(-1)
gammas = idata.posterior["gamma"].values.reshape(-1)
seleccion = rng.choice(len(betas), size=min(300, len(betas)), replace=False)
trayectorias = np.array([sir_rk4_np(betas[j], gammas[j]) for j in seleccion])
q03, mediana, q97 = np.quantile(trayectorias, [0.03, 0.5, 0.97], axis=0)

fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
for j, (nombre, color) in enumerate([("Susceptibles", "tab:blue"),
                                    ("Infectados", "tab:red"),
                                    ("Recuperados", "tab:green")]):
    ejes[j].plot(dias, mediana[:, j], color=color, label="Mediana posterior")
    ejes[j].fill_between(dias, q03[:, j], q97[:, j], color=color, alpha=.2,
                         label="Banda creíble 94%")
    ejes[j].plot(dias, solucion.y[j], "--", color="black", alpha=.7,
                 label="Trayectoria verdadera (simulada)")
    if j == 1:
        ejes[j].scatter(dias, infectados_obs, s=12, alpha=.6,
                        label="Datos observados", color="gray")
    ejes[j].set(title=nombre, xlabel="Días", ylabel="Personas")
    ejes[j].grid(alpha=.2)
ejes[1].legend(fontsize=8)
plt.suptitle("Modelo SIR: ajuste bayesiano y cuantificación de incertidumbre")
plt.tight_layout(); plt.show()

# Resumen final automático, sin inventar resultados antes de ejecutar.
for nombre, verdadero in [("beta", beta_real), ("gamma", gamma_real),
                         ("R0_basico", beta_real/gamma_real)]:
    f = resumen.loc[nombre]
    print(f"{nombre}: media={f['mean']:.4f}, HDI 94%=[{f['hdi_3%']:.4f}, {f['hdi_97%']:.4f}], referencia simulada={verdadero:.4f}")
pico_idx = np.argmax(mediana[:,1])
print(f"Pico estimado de infectados (mediana): día {int(dias[pico_idx])}, ~{mediana[pico_idx,1]:.0f} activos")

## 7. Interpretación de resultados

**Gráfica de datos:** los puntos representan observaciones de infectados activos con ruido; la curva lisa es el proceso verdadero utilizado para simularlos.

**Trazas MCMC:** cada cadena recorre sucesivamente valores de parámetros. Una mezcla visual adecuada, $\widehat R\approx1$ y tamaños efectivos suficientes apoyan la convergencia; no la garantizan por sí solos.

**Distribuciones posteriores:** muestran qué valores de $\beta$, $\gamma$ y $R_0=\beta/\gamma$ son compatibles con datos, previas y verosimilitud. Los HDI del 94% cuantifican incertidumbre paramétrica.

**Trayectorias SIR:** la curva mediana resume el ajuste; la banda indica incertidumbre sobre estados latentes, condicionada a las hipótesis del modelo. Un pico de infectados aparece cuando las nuevas infecciones dejan de superar a las recuperaciones. La dinámica depende de $\beta$, $\gamma$ y la proporción todavía susceptible.

**Criterio para concluir:** revisa la tabla de salida; verifica que los HDI cubran los valores de referencia simulados, que el ajuste represente la tendencia observada y que no existan advertencias importantes de divergencia o convergencia. Si falla alguna condición, **no declares** un ajuste satisfactorio sin investigar el motivo.

## 8. Conclusión

El enfoque bayesiano combina información previa con datos observados para estimar la incertidumbre de las tasas epidemiológicas. A diferencia de un ajuste de un único valor, PyMC produce una distribución posterior y permite construir bandas creíbles de evolución de susceptibles, infectados y recuperados. La evidencia cuantitativa final está en la tabla y las gráficas **generadas al ejecutar este cuaderno**.

**Limitaciones:** población cerrada, parámetros constantes, mezcla homogénea, condiciones iniciales conocidas, observaciones sintéticas, varianza de medición prefijada y aproximación numérica RK4. El modelo no debe emplearse para decisiones sanitarias reales sin datos apropiados y validación.

